In [3]:
# ==============================================================================
# MindKeeper - Stable version fine-tuning script
# ==============================================================================

!pip install -q -U bitsandbytes transformers peft accelerate datasets trl

import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print("⏳ [1/5] Loading standard Messages format dataset...")
from google.colab import drive
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/6895-midterm-project/data/uniformed_dementia_finetuning_dataset.jsonl'

with open(file_path, 'r', encoding='utf-8') as f:
    parsed_data = json.load(f)
dataset = Dataset.from_list(parsed_data)

print("⏳ [2/5] Loading Qwen Tokenizer & Model...")
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def format_prompt_completion(example):
    messages = example['messages']

    system_content = ""
    user_content = ""
    assistant_content = ""

    for msg in messages:
        if msg["role"] == "system":
            system_content = msg["content"]
        elif msg["role"] == "user":
            user_content = msg["content"]
        elif msg["role"] == "assistant":
            assistant_content = msg["content"]

    prompt_messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content}
    ]

    prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    try:
        parsed_content = json.loads(assistant_content)
        assistant_text = json.dumps(parsed_content, ensure_ascii=False)
    except (json.JSONDecodeError, TypeError):
        assistant_text = assistant_content if isinstance(assistant_content, str) else json.dumps(assistant_content, ensure_ascii=False)

    completion = assistant_text + tokenizer.eos_token

    return {"prompt": prompt, "completion": completion}

dataset = dataset.map(format_prompt_completion)
dataset = dataset.remove_columns(["messages"])

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map="auto"
)
model = prepare_model_for_kbit_training(model)

print("⏳ [3/5] Configuring LoRA...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print("⏳ [4/5] 🚀 Start stable version training...")
training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/6895-midterm-project/",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1.4e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    num_train_epochs=4,
    optim="adamw_torch",
    max_grad_norm=0.0,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    completion_only_loss=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=lora_config,
)

model.config.use_cache = False

trainer.train()

trainer.model.save_pretrained("/content/drive/MyDrive/6895-midterm-project/lora_weights")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/6895_project/lora_weights")
print("✅ Fine-tuning complete! Model and word segmenter saved.")

# ----------------- Test section -----------------
print("\n" + "="*60)
print("⏳ [5/5] 🧠 The Model Test in Progress: Practical Reasoning Test...")

trainer.model.eval()

test_dialogue = "Caregiver: Betty, let's sit down for dinner.\nPatient: I can't, I have to wait for John to get home from the plant. He's working the late shift."
system_prompt = "You are a professional Alzheimer's care agent. Analyze the patient's transcript, extract clinical signs, and provide an empathetic response using Validation Therapy."

messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": test_dialogue}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer(text, return_tensors="pt").to(trainer.model.device)

with torch.no_grad():
    generated_ids = trainer.model.generate(**model_inputs, max_new_tokens=512, temperature=0.1)

generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n🎯 [Perfect output of the fine-tuned model]：\n")
print(response)

⏳ [1/5] Loading standard Messages format dataset...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ [2/5] Loading Qwen Tokenizer & Model...


Map:   0%|          | 0/998 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


⏳ [3/5] Configuring LoRA...
⏳ [4/5] 🚀 Start stable version training...


...

Tokenizing train dataset:   0%|          | 0/998 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/998 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.905232
20,2.388734
30,1.856138
40,1.687503
50,1.593621
60,1.463037
70,1.411027
80,1.349163
90,1.323842
100,1.333776


✅ Fine-tuning complete! Model and word segmenter saved.

⏳ [5/5] 🧠 The Model Test in Progress: Practical Reasoning Test...

🎯 [Perfect output of the fine-tuned model]：

{"assessment": "The patient exhibits pronounced temporal disorientation by expecting a family member to return from work at an impossible hour. The reliance on a delusion (the existence of a non-existent late shift) to rationalize her current inability to prepare food indicates a moderate loss of chronological mapping.", "detected_signs": ["Temporal Confusion", "Delusion/Hallucination"], "response": "John is such a hard worker, isn't he? It is so nice that you are making sure they have enough food waiting for him when he gets home. Let's sit down and enjoy our meal while we wait.", "risk_score": 4}


In [6]:

import torch
import json

# Ensure the model is in inference mode
trainer.model.eval()

# Test Scenario Pool
test_scenarios = [
    {
        "name": "Scene 1: Temporal Distortion and Hallucinations (Searching for Deceased Relatives)",
        "dialogue": "Caregiver: Betty, let's sit down for dinner. Patient: I can't, I have to wait for John to get home from the plant. He's working the late shift."
    },
    {
        "name": "Scenario 2: Paranoia (suspecting the caregiver of stealing)",
        "dialogue": "Doctor: Good morning, Mrs. Smith. How did you sleep? Patient: I didn't sleep at all! That new nurse came in and stole my purse! I know she took it, it's missing from my drawer!"
    },
    {
        "name": "Scenario 3: Homecoming Instinct and Anxiety (Dusk Syndrome)",
        "dialogue": "Caregiver: Mr. Thomas, it's time to take your evening medication. Patient: (Pacing back and forth) No, I need to leave right now. My mother is waiting for me, and I'm going to be late for school."
    }
]

system_prompt = "You are a professional Alzheimer's care agent. Analyze the patient's transcript, extract clinical signs, and provide an empathetic response using Validation Therapy."

print("🚀 Start multi-scenario performance testing...")

for i, scenario in enumerate(test_scenarios):
    print(f"{'='*50}")
    print(f"🎬 {scenario['name']}")
    print(f"💬 Conversation log: {scenario['dialogue']}")
    print("-" * 50)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": scenario['dialogue']}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer(text, return_tensors="pt").to(trainer.model.device)

    with torch.no_grad():
        generated_ids = trainer.model.generate(
            **model_inputs,
            max_new_tokens=512,
            temperature=0.1,
            repetition_penalty=1.1
        )

    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    print("🤖 Model output (JSON):")
    try:
        parsed_json = json.loads(response)
        print(json.dumps(parsed_json, indent=4, ensure_ascii=False))

        print("✅ Format test: Perfect output JSON")
        print(f"✅ Extracting symptoms: {', '.join(parsed_json.get('detected_signs', []))}")
    except json.JSONDecodeError:
        print(response)
        print("❌ Format test: Unable to parse to standard JSON; the model may still be underfitting!")

    print(f"{'='*50}")

print("📊 [Summary of Test Conclusions]")
print("1. If the above three scenarios can consistently output JSON, and the 'response' uses empathy/affirmation therapy (without directly negating the patient), then the fine-tuning is very successful!")
print("2. Your Loss of 0.372935 is a very reasonable convergence value (no overfitting) for instruction fine-tuning and this specific task.")


🚀 Start multi-scenario performance testing...
🎬 Scene 1: Temporal Distortion and Hallucinations (Searching for Deceased Relatives)
💬 Conversation log: Caregiver: Betty, let's sit down for dinner. Patient: I can't, I have to wait for John to get home from the plant. He's working the late shift.
--------------------------------------------------
🤖 Model output (JSON):
{
    "assessment": "The patient exhibits pronounced temporal disorientation by expecting someone who is currently at work and engaged in a scheduled night-time shift. The firmness of this false belief ('I have to wait') points to a moderate level of cognitive impairment affecting reality testing.",
    "detected_signs": [
        "Temporal Confusion",
        "Delusion/Hallucination"
    ],
    "response": "John works very hard at the plant, and it is important to you that he gets home safely. You want to make sure everything is ready when he arrives. Let's keep your seat warm with some soup while we wait for him to call."

In [7]:
import torch
import json

chat_history = []

def chat_with_memory_v2(user_input: str):
    global chat_history

# Compress historical dialogues into a plain text.
    history_text = "\n".join([f"{msg['role']}: {msg['content']}" for msg in chat_history])

# Forced into the current round's Prompt
    enhanced_input = f"""[Recent Chat History]:
{history_text}

[Current Patient Input]:
{user_input}

CRITICAL: Please respond to the Current Patient Input. You MUST use facts (like your name or the person they are waiting for) from the Chat History if asked. Output ONLY JSON."""

    messages = [
        {"role": "system", "content": "You are a professional Alzheimer's care agent. Analyze the patient's transcript, extract clinical signs, and provide an empathetic response using Validation Therapy."},
        {"role": "user", "content": enhanced_input}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer(text, return_tensors="pt").to(trainer.model.device)

    with torch.no_grad():
        generated_ids = trainer.model.generate(
            **model_inputs,
            max_new_tokens=512,
            temperature=0.1
        )

    new_generated_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
    response = tokenizer.decode(new_generated_ids, skip_special_tokens=True)

# Save to history
    chat_history.append({"role": "Patient", "content": user_input})
    try:
        resp_text = json.loads(response).get("response", response)
        chat_history.append({"role": "Caregiver", "content": resp_text})
    except:
        chat_history.append({"role": "Caregiver", "content": response})

    return response

print("💬 Start the compressed memory test (V2)...")
print("-" * 40)
chat_history = []

input1 = "Caregiver: I am Sarah. Patient: My husband John is late. He works at the factory."
print(f"User: {input1}")
print(f"AI: {chat_with_memory_v2(input1)}\n")

input2 = "Patient: Wait, who are you again? and where is the person I am waiting for?"
print(f"User: {input2}")
print(f"AI: {chat_with_memory_v2(input2)}\n")


💬 Start the compressed memory test (V2)...
----------------------------------------
User: Caregiver: I am Sarah. Patient: My husband John is late. He works at the factory.
AI: {"assessment":"Patient is oriented to person and place (home/work), with no cognitive concerns.","risk_score":0}

User: Patient: Wait, who are you again? and where is the person I am waiting for?
AI: {"assessment":"The patient exhibits severe prosopagnosia by repeating the same question about their spouse's whereabouts. The inability to retain recent temporal information (waiting for John at the factory) signifies a moderate loss of continuous memory.","risk_score":4}

